In [21]:
import pytorch3d
from pytorch3d.io import load_obj
from pytorch3d.structures import Meshes
from starter.utils import get_mesh_renderer

import matplotlib.pyplot as plt

import imageio
import numpy as np
import torch

In [2]:

renderer = get_mesh_renderer(image_size=512, device="cpu")
verts, faces, aux = load_obj("data/cow.obj")

In [3]:
# set solid color texture
textures = torch.ones_like(verts) * torch.Tensor([0.5, 0.3, 0.7])

mesh = Meshes(
    verts= torch.unsqueeze(verts, 0) ,
    faces= torch.unsqueeze(faces.verts_idx, 0),
    textures=pytorch3d.renderer.TexturesVertex(torch.unsqueeze(textures, 0))
)



In [29]:
# Camera rotation stuff

# get rotation and transform matrix

my_images = []

for azimuth in range(0, 360, 20):
    R, T = pytorch3d.renderer.look_at_view_transform(3.0, 0.0, float(azimuth))
    
    cameras = pytorch3d.renderer.FoVPerspectiveCameras(
        R = R,
        T = T,
        fov=75,
    )
    
    lights = pytorch3d.renderer.PointLights(location=[[0, 0, 3]])
    
    rend = renderer(mesh, cameras=cameras, device="cpu", lights=lights)
    my_images.append(rend[0, ..., :3].numpy())

# plt.axes("off")


In [30]:
my_images_fixed = [(frame * 255).astype(np.uint8) for frame in my_images] # stretch [0., 1.] pixel value range to [0, 255] 
imageio.mimsave('cow.gif', my_images_fixed, duration=1000 // 10, loop=10)